# Day 1 — Bronze Layer Practice Questions
## Topic: HTTP requests · SQLAlchemy · PySpark ingestion · Bronze metadata

---

> **Rules:**
> - Run the setup cell first — it imports config and starts Spark
> - Solve each question before peeking at the hint
> - All data processing must use PySpark (no pandas transforms)

| # | Difficulty | Topic |
|---|-----------|-------|
| Q1 | 🟢 Easy | SQLAlchemy — inspect schemas and run raw SQL |
| Q2 | 🟡 Medium | HTTP request — fetch live weather + bronze load |
| Q3 | 🔴 Hard | PySpark — parse log file + bronze metadata pattern |

---
## Setup — run once

In [ ]:
import sys, os, json, requests
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from config.db_config import (
    get_engine, ensure_schemas, set_spark_env, get_spark,
    new_batch, raw_path, spark_write_pg
)

engine = get_engine()
ensure_schemas(engine)
BATCH_ID, INGESTED_AT = new_batch()

set_spark_env()
from pyspark.sql import functions as F

spark = get_spark('Day1_PQ')
print('Setup ready. Batch:', BATCH_ID)

---
---
## Q1 — 🟢 Easy
### SQLAlchemy: inspect the database and run a raw SQL query

---

### Problem Statement

1. Use `sqlalchemy.inspect(engine)` to list all tables in the `bronze` schema
2. Print each table name with its row count using `conn.execute(text(...))`
3. Run a raw SQL query: select the top 3 `order_id` values from `bronze.orders` sorted by `order_id`
4. Print the result rows

### Expected Output (example)
```
Tables in bronze:
  customers       50 rows
  orders          50 rows
  ...
Top 3 order_ids: ['O001', 'O002', 'O003']
```

**Hints:** `inspect(engine).get_table_names(schema='bronze')`, `conn.execute(text('SELECT COUNT(*) FROM bronze.customers'))`, `.fetchall()`

In [ ]:
# Q1 — Write your solution here


<details><summary>💡 Solution</summary>

```python
from sqlalchemy import inspect, text

insp = inspect(engine)
tables = insp.get_table_names(schema='bronze')

print('Tables in bronze:')
with engine.connect() as conn:
    for t in tables:
        n = conn.execute(text(f'SELECT COUNT(*) FROM bronze.{t}')).scalar()
        print(f'  {t:<25} {n} rows')

print()
with engine.connect() as conn:
    rows = conn.execute(
        text('SELECT order_id FROM bronze.orders ORDER BY order_id LIMIT 3')
    ).fetchall()
print('Top 3 order_ids:', [r[0] for r in rows])
```
</details>

---
---
## Q2 — 🟡 Medium
### HTTP request — fetch live weather data and load to bronze

---

### Problem Statement

Use the Open-Meteo API (free, no key needed) to fetch weather for 3 cities:
- Chicago: lat=41.8781, lon=-87.6298
- Houston: lat=29.7604, lon=-95.3698
- Seattle: lat=47.6062, lon=-122.3321

**Steps:**
1. Call `requests.get('https://api.open-meteo.com/v1/forecast', params={...})` for each city
2. Request these `current` variables: `temperature_2m,relative_humidity_2m,wind_speed_10m`
3. Build a list of dicts with: `city`, `temp_c`, `humidity_pct`, `wind_kph`, `fetched_at`
4. Create a PySpark DataFrame from the list
5. Add bronze metadata columns using `F.lit()`
6. Write to `bronze.pq_live_weather` using `spark_write_pg(df, 'bronze', 'pq_live_weather')`
7. Print the row count

### Expected Output
```
Fetched: 3 cities
  bronze.pq_live_weather → 3 rows
```

**Hints:** `resp.raise_for_status()`, `resp.json()['current']`, `spark.createDataFrame(list_of_dicts)`

In [ ]:
# Q2 — Write your solution here


<details><summary>💡 Solution</summary>

```python
from datetime import datetime

CITIES = [
    {'name': 'Chicago', 'lat': 41.8781, 'lon': -87.6298},
    {'name': 'Houston', 'lat': 29.7604, 'lon': -95.3698},
    {'name': 'Seattle', 'lat': 47.6062, 'lon': -122.3321},
]

records = []
for city in CITIES:
    resp = requests.get(
        'https://api.open-meteo.com/v1/forecast',
        params={
            'latitude' : city['lat'],
            'longitude': city['lon'],
            'current'  : 'temperature_2m,relative_humidity_2m,wind_speed_10m',
            'timezone' : 'UTC',
        },
        timeout=10
    )
    resp.raise_for_status()
    cur = resp.json()['current']
    records.append({
        'city'        : city['name'],
        'temp_c'      : cur['temperature_2m'],
        'humidity_pct': cur['relative_humidity_2m'],
        'wind_kph'    : cur['wind_speed_10m'],
        'fetched_at'  : INGESTED_AT,
    })
    print(f'  {city["name"]}: {cur["temperature_2m"]}°C')

print(f'Fetched: {len(records)} cities')

df_w = (spark.createDataFrame(records)
        .withColumn('_source_file', F.lit('open-meteo-api'))
        .withColumn('_ingested_at', F.lit(INGESTED_AT))
        .withColumn('_batch_id',    F.lit(BATCH_ID)))

spark_write_pg(df_w, 'bronze', 'pq_live_weather')
df_w.show(truncate=False)
```
</details>

---
---
## Q3 — 🔴 Hard
### PySpark — parse a log file and apply the bronze metadata pattern

---

### Problem Statement

Parse `data/raw/webserver_access.log` using simple string splitting (no regex), then:

1. Open the file and read it line by line
2. For each line, split on `"` (double-quote) to get 5 parts
3. Extract fields from the parts — see the table below:

| Field | How to get it |
|-------|--------------|
| `ip` | `parts[0].split()[0]` |
| `user` | `parts[0].split()[2]` — use `None` if `'-'` |
| `timestamp` | `parts[0].split()[3]` — strip leading `[` |
| `method` | `parts[1].split()[0]` |
| `endpoint` | `parts[1].split()[1]` |
| `status_code` | `parts[2].strip().split()[0]` cast to `int` |
| `response_size` | `parts[2].strip().split()[1]` cast to `int` — `None` if not digit |
| `referrer` | `parts[3]` — `None` if `'-'` |
| `user_agent` | `parts[4].strip()` |

4. Create a **PySpark** DataFrame from the list of dicts
5. Add bronze metadata: `_source_file`, `_ingested_at`, `_batch_id` using `F.lit()`
6. Count rows by `status_code` using `.groupBy().count()`
7. Write to `bronze.pq_web_logs` using `spark_write_pg(df, 'bronze', 'pq_web_logs')`
8. Print the status distribution

**Hints:** `line.split('"')` gives 5 parts, `parts[0].strip().split()` gives individual tokens

In [ ]:
# Q3 — Write your solution here


<details><summary>💡 Solution</summary>

```python
from pyspark.sql.types import IntegerType

rows = []
with open(raw_path('webserver_access.log')) as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        parts  = line.split('"')
        left   = parts[0].strip().split()
        req    = parts[1].split()
        middle = parts[2].strip().split()
        rows.append({
            'ip'           : left[0],
            'user'         : left[2] if left[2] != '-' else None,
            'timestamp'    : left[3].lstrip('['),
            'method'       : req[0] if req else None,
            'endpoint'     : req[1] if len(req) > 1 else None,
            'status_code'  : int(middle[0]) if middle else None,
            'response_size': int(middle[1]) if len(middle) > 1 and middle[1].isdigit() else None,
            'referrer'     : parts[3] if parts[3] != '-' else None,
            'user_agent'   : parts[4].strip(),
            'line_num'     : i,
        })

print(f'Parsed: {len(rows)} rows')

df_logs = (
    spark.createDataFrame(rows)
    .withColumn('status_code', F.col('status_code').cast(IntegerType()))
    .withColumn('_source_file', F.lit('webserver_access.log'))
    .withColumn('_ingested_at', F.lit(INGESTED_AT))
    .withColumn('_batch_id',    F.lit(BATCH_ID))
)

print('Status distribution:')
df_logs.groupBy('status_code').count().orderBy('status_code').show()

spark_write_pg(df_logs, 'bronze', 'pq_web_logs')
print('bronze.pq_web_logs loaded.')
```
</details>

In [ ]:
spark.stop()
print('Spark stopped.')